In [ ]:
"""
Strategi Finetuning.
  Grid Search
  Random Search
"""

In [ ]:
"""
Grid search:
  Mencoba segala kombinasi hyper paramter dari range atau nilai yg ditentukan (semacam bruteforce lah).
  Misal jika kita punya Hyper Pramter:
    {
      "learning_rate": [0.01, 0.001, 0.0001],
      "n_neoron": [3, 6, 9]
    }
  Maka kan dicoba berbagai kombinasi menjadi:
    [
      [0.01, 3],
      [0.01, 6],
      [0.01, 9],
      [0.001, 3],
      [0.001, 6],
      [0.001, 9],
      [0.0001, 3],
      [0.0001, 6],
      [0.0001, 9],
    ]
"""

'\nGrid search:\n  Mencoba segala kombinasi hyper paramter dari range atau nilai yg ditentukan (semacam bruteforce lah).\n  Misal jika kita punya Hyper Pramter:\n    {\n      "learning_rate": [0.01, 0.001, 0.0001],\n      "n_neoron": [3, 6, 9] \n    }\n  Maka kan dicoba berbagai kombinasi menjadi:\n'

In [ ]:
import numpy as np
np.logspace(1, 2, 9, -4)

array([ 10.        ,  13.33521432,  17.7827941 ,  23.71373706,
        31.6227766 ,  42.16965034,  56.23413252,  74.98942093,
       100.        ])

In [ ]:
# Studi kasus
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = pd.read_csv("./drive/MyDrive/documents/BelajarML/harga_rumah_250.csv")
y = data['harga_juta']
x = data.drop(columns='harga_juta')
scaler = StandardScaler()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.2, random_state=42)
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

print(f"""
x_train: {x_train.shape}
x_test: {x_test.shape}
""")


x_train: (212, 4)
x_test: (54, 4)



In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

rf = RandomForestRegressor(random_state=42)

# Evaluasi awal tanpa tuning
rf.fit(x_train, y_train)
initial_predict = rf.predict(x_test)
initial_mse = mean_squared_error(y_test, initial_predict)
print(f"MSE awal tanpa tuning: {initial_mse}")

MSE awal tanpa tuning: 7493.732343421936


In [ ]:
import time
from sklearn.model_selection import GridSearchCV
start = time.time()

param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'bootstrap': [True, False]
}

grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(x_train, y_train)

end = time.time()
print(f"Waktu yang dibutuhkan: {end - start}")

Fitting 3 folds for each of 384 candidates, totalling 1152 fits
Waktu yang dibutuhkan: 176.66546964645386


In [ ]:

# Output hasil terbaik
print(f"Best parameters (Grid Search): {grid_search.best_params_}")
best_rf_grid = grid_search.best_estimator_

# Evaluasi performa model setelah Grid Search
y_pred_grid = best_rf_grid.predict(x_test)
grid_search_mse = mean_squared_error(y_test, y_pred_grid)
print(f"MSE after Grid Search: {grid_search_mse:.2f}")

Best parameters (Grid Search): {'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
MSE after Grid Search: 2.44
Waktu eksekusi: 1627.6728 detik


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

start = time.time()
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_grid, n_iter=4, cv=3, n_jobs=-1, verbose=2)
random_search.fit(x_train, y_train)
end = time.time()

best_params = random_search.best_params_
best_model = random_search.best_estimator_
preds = best_model.predict(x_test)
mse = mean_squared_error(y_test, preds)

print(f"""
Waktu yang dibutuhkan: {end - start}
Best parameter (Random Search): {best_params}
MSE (Random Search): {mse}
""")

Fitting 3 folds for each of 4 candidates, totalling 12 fits

Waktu yang dibutuhkan: 1.7828443050384521
Best parameter (Random Search): {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 10, 'bootstrap': False}
MSE (Random Search): 9258.336940586403



In [ ]:
%pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 6.3 MB/s eta 0:00:00


In [ ]:
from skopt import BayesSearchCV

start = time.time()
bayes_search = BayesSearchCV(
    estimator=rf,
    search_spaces=param_grid,
    n_iter=4,
    cv=3,
    n_jobs=-1,
    verbose=2
)
bayes_search.fit(x_train, y_train)
end = time.time()

best_params = bayes_search.best_params_
best_model = bayes_search.best_estimator_
preds = best_model.predict(x_test)
mse = mean_squared_error(y_test, preds)

print(f"""
Waktu yang dibutuhkan: {end - start}
best params (Bayes Search): {best_params}
MSE (Bayes Search): {mse}
""")

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits

Waktu yang dibutuhkan: 4.549820899963379
best params (Bayes Search): OrderedDict({'bootstrap': True, 'max_depth': 30, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100})
MSE (Bayes Search): 7218.594460575741

